In [1]:
import numpy as np
import pandas as pd

In [2]:
from pykrx import stock
import datetime

# 삼성전자 티커: 005930
ticker = "005930"
today = datetime.datetime.today().strftime("%Y%m%d")
# 최근 7일간의 시세 조회
start_date = (datetime.datetime.today() - datetime.timedelta(days=7)).strftime("%Y%m%d")

print(f"조회 기간: {start_date} ~ {today}")
df = stock.get_market_ohlcv_by_date(start_date, today, ticker)
print(df)

KRX 로그인 실패: KRX_ID 또는 KRX_PW 환경 변수가 설정되지 않았습니다.
조회 기간: 20260818 ~ 20260825
                시가      고가      저가      종가       거래량       등락률
날짜                                                            
2026-08-18  283000  288000  265000  268500  24464621 -2.185792
2026-08-19  251500  254500  246500  247500  22788552 -7.821229
2026-08-20  257000  273000  252500  271000  26095919  9.494949
2026-08-21  267000  285000  266000  281500  27746471  3.874539
2026-08-24  271500  272000  255000  257000  32451480 -8.703375


In [3]:
# 네이버 뉴스 검색 API (https://developers.naver.com/docs/serviceapi/search/news/news.md)
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from email.utils import parsedate_to_datetime
from html import unescape

load_dotenv()

# 환경변수에서 API 키 읽기 — .env 파일 또는 시스템 환경변수 사용
# 절대 코드에 직접 값을 적지 마세요 (GitHub에 올라가면 키가 노출됩니다)
CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

url = "https://openapi.naver.com/v1/search/news.json"
headers = {
    "X-Naver-Client-Id": CLIENT_ID or "",
    "X-Naver-Client-Secret": CLIENT_SECRET or "",
}

def search_news(query: str, display: int = 20, start: int = 1, sort: str = "date") -> pd.DataFrame:
    """네이버 뉴스 검색 결과를 DataFrame으로 반환. sort: date(최신순) | sim(정확도순)"""
    if not CLIENT_ID or not CLIENT_SECRET:
        raise RuntimeError(".env에 NAVER_CLIENT_ID와 NAVER_CLIENT_SECRET을 설정하세요.")

    params = {"query": query, "display": display, "start": start, "sort": sort}
    resp = requests.get(url, headers=headers, params=params, timeout=10)
    resp.raise_for_status()
    items = resp.json().get("items", [])
    rows = [
        {
            "title": unescape(item["title"].replace("<b>", "").replace("</b>", "")),
            "description": unescape(item["description"].replace("<b>", "").replace("</b>", "")),
            "link": item["link"],
            "originallink": item["originallink"],
            "pubDate": parsedate_to_datetime(item["pubDate"]),
        }
        for item in items
    ]
    return pd.DataFrame(rows)

# 관련 뉴스 최근 20건
if CLIENT_ID and CLIENT_SECRET:
    news_df = search_news("트럼프 정부", display=20, sort="date")
    print(f"수집된 뉴스 수: {len(news_df)}")
    display(news_df.head(10))
else:
    news_df = pd.DataFrame()
    print(".env에 NAVER_CLIENT_ID와 NAVER_CLIENT_SECRET을 설정하세요.")

수집된 뉴스 수: 20


,title,description,link,originallink,pubDate
0,"오피셜 트럼프(TRUMP), 일주일 만에 80% 급등...93% 폭락 충격 지우기 ...",공식 웹사이트는 해당 토큰이 트럼프 상징이 나타내는 가치와 신념을 지지하는 상징적 ...,http://coinreaders.com/254539,http://coinreaders.com/254539,2026-08-25 01:32:00+09:00
1,'연간 8.5만명 추첨' 미국 H-1B 비자... 발급 수수료 1.4억원 추가될 듯,트럼프 행정부는 인도와 중국인 비중이 높은 H-1B 프로그램이 미국 졸업생들의 취업...,https://n.news.naver.com/mnews/article/469/000...,https://www.hankookilbo.com/news/article/A2026...,2026-08-25 01:30:00+09:00
2,푸드스탬프 수혜자 560만명 급감,도널드 트럼프 대통령의 이른바 ‘하나의 크고 아름다운 법안’에 따른 근로 요건 강화...,http://www.koreatimes.com/article/1626848,http://www.koreatimes.com/article/1626848,2026-08-25 01:26:00+09:00
3,"美 전문직 취업비자 문턱 높인다… ""수수료 1억4000만원 제도화""",AFP 연합뉴스 도널드 트럼프 미국 행정부가 외국인 전문직 근로자들이 주로 이용하는...,https://www.dkilbo.com/news/articleView.html?i...,https://www.dkilbo.com/news/articleView.html?i...,2026-08-25 01:14:00+09:00
4,"미·캐나다 관세전쟁 장기화?…캐나다, 기업 지원책 마련","블룸버그통신은 현지시간 23일 소식통들을 인용해, 마크 카니 캐나다 정부가 중간선거...",https://n.news.naver.com/mnews/article/056/001...,https://news.kbs.co.kr/news/pc/view/view.do?nc...,2026-08-25 01:12:00+09:00
5,"한미, 대북공조 확인했지만 '비핵화' 명시 없이 '북핵문제 해결'",특히 정부는 트럼프 집권 1기 때와 달리 남북 대화가 단절돼 이전처럼 직접 북미 대...,https://n.news.naver.com/mnews/article/001/001...,https://www.yna.co.kr/view/AKR2026082500130050...,2026-08-25 01:11:00+09:00
6,"조현, 루비오와 통화…“韓·美 긴밀소통” 강조",조 장관은 최근 도널드 트럼프 미국 대통령의 메시지가 북미 대화 재개와 한반도 평화...,https://n.news.naver.com/mnews/article/366/000...,https://biz.chosun.com/international/internati...,2026-08-25 01:06:00+09:00
7,"트럼프정부 ""전문직 비자에 1.4억 수수료""…법원 제동에도 강행(종합)",다만 트럼프 행정부는 미국 대학을 졸업한 외국인 유학생이 미국에서 취업할 때 10만...,https://n.news.naver.com/mnews/article/001/001...,https://www.yna.co.kr/view/AKR2026082416725107...,2026-08-25 00:58:00+09:00
8,"조현, 美국무장관과 통화…“한반도 평화·북핵 해결 긴밀 공조”",조 장관은 최근 도널드 트럼프 대통령의 메시지가 북미 대화 재개와 한반도 평화로 이...,https://n.news.naver.com/mnews/article/021/000...,https://www.munhwa.com/article/11611687?ref=naver,2026-08-25 00:54:00+09:00
9,트럼프 “비트코인 추가매입 논의했다”면서도 구체 시점·조달방식은 ...,이날 회동은 워싱턴이 디지털자산 관련 입법과 규제 관할권을 정리해가는 과정에서 열렸...,https://www.wikitree.co.kr/articles/1154615,https://www.wikitree.co.kr/articles/1154615,2026-08-25 00:52:00+09:00


In [4]:
# Grok(xAI) API 연결 테스트 (https://docs.x.ai/)
import os
import requests as _requests
from dotenv import load_dotenv

load_dotenv(override=True)

# 환경변수에서 API 키 읽기 (콘솔: https://console.x.ai)
# 절대 코드에 직접 값을 적지 마세요 (GitHub에 올라가면 키가 노출됩니다)
XAI_API_KEY = os.getenv("XAI_API_KEY")

GROK_URL = "https://api.x.ai/v1/chat/completions"
GROK_MODEL = "grok-4-1-fast-non-reasoning"

def ask_grok(prompt: str, system: str = "You are a helpful assistant.", temperature: float = 0.7) -> str:
    """Grok 챗 컴플리션 호출 후 응답 텍스트 반환"""
    if not XAI_API_KEY:
        raise RuntimeError(".env에 XAI_API_KEY를 설정하세요.")

    resp = _requests.post(
        GROK_URL,
        headers={
            "Authorization": f"Bearer {XAI_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": GROK_MODEL,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
            ],
            "temperature": temperature,
        },
        timeout=60,
    )
    if not resp.ok:
        raise RuntimeError(f"xAI API 오류 ({resp.status_code}): {resp.text}")
    return resp.json()["choices"][0]["message"]["content"]

# 간단 연결 테스트
if XAI_API_KEY:
    try:
        print(ask_grok("한 문장으로 인사말을 해줘."))
    except RuntimeError as error:
        print(error)
else:
    print(".env에 XAI_API_KEY를 설정하세요.")

안녕하세요!


In [5]:
# 한국거래소(KRX) OpenAPI — 시가총액 상위 10개 종목 (https://openapi.krx.co.kr)
import os
import requests as _requests2
import pandas as _pd2
from dotenv import load_dotenv

load_dotenv(override=True)

# 환경변수에서 인증키 읽기 — .env 파일 또는 시스템 환경변수 사용
# 절대 코드에 직접 값을 적지 마세요 (GitHub에 올라가면 키가 노출됩니다)
# 발급: openapi.krx.co.kr → 마이페이지 → API 인증키 신청
# 주의: 키 발급 후 "서비스 이용 → 주식 → 유가증권 일별매매정보"의 API 이용신청도 필요 (승인 약 1일)
KRX_AUTH_KEY = os.getenv("KRX_AUTH_KEY")

KRX_BASE_URL = "https://data-dbg.krx.co.kr/svc/apis"

def get_krx_daily_trade(bas_dd: str, market: str = "stk") -> _pd2.DataFrame:
    """일자별 전종목 매매정보 조회. market: stk(코스피) | ksq(코스닥)"""
    if not KRX_AUTH_KEY:
        raise RuntimeError(".env에 KRX_AUTH_KEY를 설정하세요.")

    url = f"{KRX_BASE_URL}/sto/{market}_bydd_trd"
    resp = _requests2.get(url, params={"AUTH_KEY": KRX_AUTH_KEY, "basDd": bas_dd}, timeout=30)
    resp.raise_for_status()
    return _pd2.DataFrame(resp.json().get("OutBlock_1", []))

# 최근 영업일 기준 전체 종목 시세 → 시가총액 상위 10개 필터링
if KRX_AUTH_KEY:
    krx_df = get_krx_daily_trade("20260821")
    print(f"조회된 종목 수: {len(krx_df)}")
    required_columns = {"ISU_CD", "ISU_NM", "MKTCAP"}
    if not required_columns.issubset(krx_df.columns):
        raise RuntimeError(f"KRX 응답에 필요한 컬럼이 없습니다: {required_columns}")

    krx_df["MKTCAP"] = _pd2.to_numeric(krx_df["MKTCAP"], errors="coerce")
    top_10 = krx_df.nlargest(10, "MKTCAP")
    display(top_10[["ISU_CD", "ISU_NM", "MKT_NM", "TDD_CLSPRC", "FLUC_RT", "MKTCAP"]])
else:
    krx_df = _pd2.DataFrame()
    print(".env에 KRX_AUTH_KEY를 설정하세요.")

조회된 종목 수: 942


,ISU_CD,ISU_NM,MKT_NM,TDD_CLSPRC,FLUC_RT,MKTCAP
457,005930,삼성전자,KOSPI,281500,3.87,1645727428152000
177,000660,SK하이닉스,KOSPI,1730000,2.31,1263751791450000
458,005935,삼성전자우,KOSPI,207000,8.26,166090839021000
165,402340,SK스퀘어,KOSPI,1123000,0.00,148150649754000
455,009150,삼성전기,KOSPI,1316000,-5.73,98296903936000
906,005380,현대차,KOSPI,415000,-0.60,84974472890000
104,373220,LG에너지솔루션,KOSPI,343500,-4.05,80379000000000
451,207940,삼성바이오로직스,KOSPI,1551000,-1.46,71797265001000
452,032830,삼성생명,KOSPI,328500,10.61,65700000000000
449,028260,삼성물산,KOSPI,395500,5.75,64137278285500


In [6]:
# NVIDIA NIM API 연결 테스트 (https://build.nvidia.com)
import os
import requests as _nvidia_requests
from dotenv import load_dotenv

load_dotenv(override=True)

# .env에 NVIDIA_API_KEY=발급받은_키 또는 NVIDIABuild-Autogen=발급받은_키 형식으로 저장하세요.
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or os.getenv("NVIDIABuild-Autogen")
NVIDIA_URL = "https://integrate.api.nvidia.com/v1/chat/completions"
NVIDIA_MODEL = "meta/llama-3.1-8b-instruct"

def ask_nvidia(prompt: str, system: str = "You are a helpful assistant.") -> str:
    """NVIDIA NIM 챗 컴플리션 호출 후 응답 텍스트 반환"""
    if not NVIDIA_API_KEY:
        raise RuntimeError(".env에 NVIDIA_API_KEY 또는 NVIDIABuild-Autogen을 설정하세요.")

    resp = _nvidia_requests.post(
        NVIDIA_URL,
        headers={
            "Authorization": f"Bearer {NVIDIA_API_KEY}",
            "Accept": "application/json",
            "Content-Type": "application/json",
        },
        json={
            "model": NVIDIA_MODEL,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
            ],
            "temperature": 0.7,
            "max_tokens": 256,
        },
        timeout=60,
    )
    if not resp.ok:
        raise RuntimeError(f"NVIDIA API 오류 ({resp.status_code}): {resp.text}")
    return resp.json()["choices"][0]["message"]["content"]

# 간단 연결 테스트
if NVIDIA_API_KEY:
    try:
        print(ask_nvidia("한 문장으로 인사말을 해줘."))
    except RuntimeError as error:
        print(error)
else:
    print(".env에 NVIDIA_API_KEY 또는 NVIDIABuild-Autogen을 설정하세요.")

안녕하세요, 질문이나 도움이 필요하신다면 언제든지 말씀해 주세요.


In [7]:
# 한국투자증권(KIS) API 연결 테스트 — 접근 토큰(access token) 발급 (https://apiportal.koreainvestment.com)
import os
import json
import time
from pathlib import Path
import requests as _kis_requests
from dotenv import load_dotenv

load_dotenv(override=True)

# 환경변수에서 앱키/시크릿 읽기 — .env 파일 또는 시스템 환경변수 사용
# 절대 코드에 직접 값을 적지 마세요 (GitHub에 올라가면 키가 노출됩니다)
KIS_APP_KEY = os.getenv("KIS_APP_KEY")
KIS_APP_SECRET = os.getenv("KIS_APP_SECRET")
KIS_ACCOUNT_NO = os.getenv("KIS_ACCOUNT_NO")

# 실전투자: openapi.koreainvestment.com:9443 | 모의투자: openapivts.koreainvestment.com:29443
KIS_BASE_URL = os.getenv("KIS_BASE_URL", "https://openapi.koreainvestment.com:9443")

# 토큰 발급은 1분당 1회로 제한되어 있어 파일에 캐싱해서 재사용한다 (.gitignore 처리됨)
KIS_TOKEN_CACHE_PATH = Path(".kis_token_cache.json")


def _load_cached_kis_token() -> str | None:
    if not KIS_TOKEN_CACHE_PATH.exists():
        return None
    try:
        cache = json.loads(KIS_TOKEN_CACHE_PATH.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return None
    if cache.get("app_key") != KIS_APP_KEY:
        return None
    if time.time() >= cache.get("expires_at", 0):
        return None
    return cache.get("access_token")


def _save_kis_token_cache(token: str, expires_in: int) -> None:
    cache = {
        "app_key": KIS_APP_KEY,
        "access_token": token,
        "expires_at": time.time() + expires_in - 300,  # 5분 여유를 두고 만료 처리
    }
    KIS_TOKEN_CACHE_PATH.write_text(json.dumps(cache), encoding="utf-8")


def get_kis_access_token(force_refresh: bool = False) -> str:
    """앱키/시크릿으로 접근 토큰을 발급받는다 (유효기간 약 24시간, 파일 캐싱으로 재사용)."""
    if not KIS_APP_KEY or not KIS_APP_SECRET:
        raise RuntimeError(".env에 KIS_APP_KEY와 KIS_APP_SECRET을 설정하세요.")

    if not force_refresh:
        cached = _load_cached_kis_token()
        if cached:
            return cached

    resp = _kis_requests.post(
        f"{KIS_BASE_URL}/oauth2/tokenP",
        headers={"Content-Type": "application/json"},
        json={
            "grant_type": "client_credentials",
            "appkey": KIS_APP_KEY,
            "appsecret": KIS_APP_SECRET,
        },
        timeout=20,
    )
    if not resp.ok:
        raise RuntimeError(f"KIS 토큰 발급 오류 ({resp.status_code}): {resp.text}")

    payload = resp.json()
    token = payload["access_token"]
    _save_kis_token_cache(token, payload.get("expires_in", 86400))
    return token


# 연결 테스트: 토큰 발급(또는 캐시 재사용) 성공 여부만 확인
if KIS_APP_KEY and KIS_APP_SECRET:
    try:
        was_cached = _load_cached_kis_token() is not None
        token = get_kis_access_token()
        source = "캐시 재사용" if was_cached else "신규 발급"
        print(f"토큰 준비 완료 ({source}, 길이 {len(token)}자): {token[:10]}...")
        print(f"계좌번호 설정: {KIS_ACCOUNT_NO}")
    except RuntimeError as error:
        print(error)
else:
    print(".env에 KIS_APP_KEY와 KIS_APP_SECRET을 설정하세요.")

토큰 준비 완료 (캐시 재사용, 길이 346자): eyJ0eXAiOi...
계좌번호 설정: 63133047-01
